# Keras GPU Smoke Test

Notebook simples para validar o ambiente com `keras`, visualizar algumas imagens e treinar uma rede pequena.

In [24]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CACHE_DIR = PROJECT_ROOT / ".cache"
MPL_CACHE_DIR = CACHE_DIR / "matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPL_CACHE_DIR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from PIL import Image
from tensorflow import keras

In [25]:
SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

tf.keras.utils.set_random_seed(SEED)

DATA_DIR = PROJECT_ROOT / "data" / "faces"
IMAGES_DIR = DATA_DIR / "final_files"
LABELS_PATH = DATA_DIR / "labels.csv"
GPUS = tf.config.list_physical_devices("GPU")
GPU_STATUS = "OK" if GPUS else "Nao detectada pelo TensorFlow"

for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

print("Projeto:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU status:", GPU_STATUS)
print("GPUs:", GPUS)
print("Labels:", LABELS_PATH.exists(), LABELS_PATH)
print("Imagens:", IMAGES_DIR.exists(), IMAGES_DIR)

Projeto: /home/f_szekut/projects/TT/Keras_ageid
TensorFlow: 2.21.0
Keras: 3.14.0
GPU status: OK
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Labels: True /home/f_szekut/projects/TT/Keras_ageid/data/faces/labels.csv
Imagens: True /home/f_szekut/projects/TT/Keras_ageid/data/faces/final_files


In [26]:
def build_image_path(file_name: object) -> str | None:
    if pd.isna(file_name):
        return None
    return str(IMAGES_DIR / str(file_name))

labels = pd.read_csv(LABELS_PATH)
labels["path"] = labels["file_name"].map(build_image_path)
labels = labels[labels["path"].notna()].copy()
labels = labels[labels["path"].map(lambda p: Path(str(p)).exists())].copy()
labels["age_group"] = (labels["real_age"] >= 30).astype("int32")

print(labels.head())
print("\nTotal de imagens:", len(labels))
print("Distribuicao da classe age_group:")
print(labels["age_group"].value_counts().sort_index())

    file_name  real_age                                               path  \
0  000000.jpg         4  /home/f_szekut/projects/TT/Keras_ageid/data/fa...   
1  000001.jpg        18  /home/f_szekut/projects/TT/Keras_ageid/data/fa...   
2  000002.jpg        80  /home/f_szekut/projects/TT/Keras_ageid/data/fa...   
3  000003.jpg        50  /home/f_szekut/projects/TT/Keras_ageid/data/fa...   
4  000004.jpg        17  /home/f_szekut/projects/TT/Keras_ageid/data/fa...   

   age_group  
0          0  
1          0  
2          1  
3          1  
4          0  

Total de imagens: 7591
Distribuicao da classe age_group:
age_group
0    3951
1    3640
Name: count, dtype: int64


In [ ]:
sample_df = labels.sample(9, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, row in zip(axes.flatten(), sample_df.itertuples()):
    image = Image.open(Path(str(row.path)))
    ax.imshow(image)
    ax.set_title(f"idade={row.real_age} | grupo={row.age_group}")
    ax.axis("off")

plt.tight_layout()

In [ ]:
work_df = labels.sample(min(len(labels), 1200), random_state=SEED).reset_index(drop=True)
split_index = int(len(work_df) * 0.8)
train_df = work_df.iloc[:split_index].copy()
val_df = work_df.iloc[split_index:].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

@tf.autograph.experimental.do_not_convert
def load_example(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["age_group"].values))
    dataset = dataset.map(load_example, num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(*IMAGE_SIZE, 3)),
    keras.layers.Conv2D(16, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation="relu"),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=2
)

In [ ]:
pd.DataFrame(history.history)